In [22]:
import sys
from pathlib import Path

from autostorage import Database, query
from autostorage.database import CalculationRow, EnergyRow, GeometryRow, GradientRow, HessianRow, ModelRow, StationaryPointRow, TrajectoryRow

sys.path.insert(0, str(Path.cwd()))

import helpers  # ty:ignore[unresolved-import]imp

FILE_DIR = Path.cwd() / "data" / "orca_stationary"

db = Database("demo.db")

goat_model = ModelRow(
    program="orca", program_version="6.1.1", method="xtb", calc_type="goat"
)
# Query for existing match
goat_model = query.first_match(db, goat_model) or goat_model

init_geo = GeometryRow.from_xyz_file(FILE_DIR / "init.xyz", charge=0, spin=1)
# Special query for geometries (looks for hash)
init_geo = query.geometry_match(db, init_geo) or init_geo

goat_calc = CalculationRow(
    model=goat_model,
    input_geometry=init_geo,
    input_provenance={"input file": (FILE_DIR / "goat.inp").read_text()}
)
# Query for existing match
goat_calc = query.first_match(db, goat_calc) or goat_calc

if not goat_calc.id:
    min_geo = GeometryRow.from_xyz_file(
        FILE_DIR / "goat.xyz",
        # We reference the charge and spin of the initial geometry for consistency
        charge=init_geo.charge,
        spin=init_geo.spin
    )
    min_geo = query.geometry_match(db, min_geo) or min_geo
    goat_calc.output_geometry = min_geo

    ensemble = TrajectoryRow.from_xyz_file(
        FILE_DIR / "goat.finalensemble.xyz", charge=init_geo.charge, spin=init_geo.spin
    )
    goat_calc.output_trajectory = ensemble

    # The goat_calc contains the inputs / outputs which will be inserted alongside the calculation
    db.add(goat_calc)

In [23]:
opt_model = ModelRow(
    program="orca", program_version="6.1.1", method="wb97x-3c", calc_type="opt"
)
# Query for existing match
opt_model = query.first_match(db, opt_model) or opt_model

opt_calc = CalculationRow(
    model=opt_model,
    input_geometry=goat_calc.output_geometry,
    input_provenance={"input file": (FILE_DIR / "opt.inp").read_text()}
)
# Query for existing match
opt_calc = query.first_match(db, opt_calc) or opt_calc

if opt_calc.id is None:
    opt_geo = GeometryRow.from_xyz_file(
        FILE_DIR / "opt.xyz",
        # We reference the charge and spin of the initial geometry for consistency
        charge=init_geo.charge,
        spin=init_geo.spin,
    )
    opt_geo = query.geometry_match(db, opt_geo) or opt_geo
    opt_calc.output_geometry = opt_geo

    db.add(opt_calc)

    # Optimized geometries are stored as a stationary point to indicate they are optimized
    opt_stp = StationaryPointRow(
        geometry=opt_geo,
        calculation=opt_calc,
    )
    # We don't need to query here because if the opt_calc.id is None there won't be a matching stationary point
    db.add(opt_stp)

/home/tns97255/coding/autodev/autostorage/.pixi/envs/dev/lib/python3.14/site-packages/sqlmodel/orm/session.py:75: SAWarning: Object of type <CalculationRow> not in session, add operation along 'GeometryRow.calculation_inputs' will not proceed (This warning originated from the Session 'autoflush' process, which was invoked automatically in response to a user-initiated operation. Consider using ``no_autoflush`` context manager if this warning happened while initializing objects.)
  results = super().execute(


In [24]:
freq_model = ModelRow(
    program="orca", program_version="6.1.1", method="m062x", basis="def2-TZVPP", calc_type="freq"
)
# Query for existing match
freq_model = query.first_match(db, freq_model) or freq_model

freq_calc = CalculationRow(
    model=freq_model,
    input_geometry=opt_calc.output_geometry,
    input_provenance={"input file": (FILE_DIR / "freq.inp").read_text()}
)
# Query for existing match
freq_calc = query.first_match(db, freq_calc) or freq_calc

if freq_calc.id is None:
    freq_geo = GeometryRow.from_xyz_file(
        FILE_DIR / "freq.xyz",
        # We reference the charge and spin of the initial geometry for consistency
        charge=init_geo.charge,
        spin=init_geo.spin,
    )
    freq_geo = query.geometry_match(db, freq_geo) or freq_geo
    freq_calc.output_geometry = freq_geo

    db.add(freq_calc)

    freq_grad = GradientRow(
        geometry=freq_geo,
        calculation=freq_calc,
        value=helpers.orca_parse_gradient((FILE_DIR / "freq.engrad").read_text())
    )

    db.add(freq_grad)

    freq_hess = HessianRow(
        geometry=freq_geo,
        calculation=freq_calc,
        value=helpers.orca_parse_hessian((FILE_DIR / "freq.hess").read_text())
    )

    # Optimized geometries are stored as a stationary point to indicate they are freqimized
    freq_stp = StationaryPointRow(
        geometry=freq_geo,
        calculation=freq_calc,
        # We pass the hessian along with the stationary point row to link them and verify the order
        hessian=freq_hess,
    )
    # We don't need to query here because if the freq_calc.id is None there won't be a matching stationary point
    db.add(freq_stp)

In [25]:
energy_model = ModelRow(
    program="orca", program_version="6.1.1", method="CCSD(T)-F12", basis="cc-pVDZ-F12", calc_type="energy"
)
# Query for existing match
energy_model = query.first_match(db, energy_model) or energy_model

energy_calc = CalculationRow(
    model=energy_model,
    input_geometry=freq_calc.output_geometry,
    input_provenance={"input file": (FILE_DIR / "energy.inp").read_text()}
)
# Query for existing match
energy_calc = query.first_match(db, energy_calc) or energy_calc

if energy_calc.id is None:
    db.add(energy_calc)

    energy = EnergyRow(
        geometry=energy_calc.input_geometry,
        calculation=energy_calc,
        value=helpers.orca_parse_spe((FILE_DIR / "energy.log").read_text())
    )

    db.add(energy)